In [4]:
import pandas as pd
import geohash2

In [15]:
ais_data_df=pd.read_csv("AIS_data_small.csv")
ships_small_df=pd.read_csv("ships_small.csv")
radio_signatures_df=pd.read_csv("radio_signatures_small.csv")

In [16]:
ais_data_df['geohash'] = ais_data_df.apply(
    lambda row: geohash2.encode(row['latitude'], row['longitude'], precision=8), 
    axis=1
)
ais_data_df = ais_data_df.drop(['latitude', 'longitude'], axis=1)

In [17]:
df_merged = ais_data_df.merge(
    ships_small_df[['mmsi', 'name', 'type', 'flag', 'destination']], 
    on='mmsi', 
    how='left'          # 'left', 'inner', 'right', ou 'outer'
)
df_merged = df_merged.merge(radio_signatures_df, on='mmsi', how='left')

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

df_prep = df_merged.copy()

# ====================== 2. GESTION DES TIMESTAMPS ======================
# Unifier les timestamps
df_prep['timestamp'] = pd.to_datetime(df_prep['timestamp_x']).fillna(
                       pd.to_datetime(df_prep['timestamp_y']))

# Extraire des features temporelles utiles
df_prep['hour'] = df_prep['timestamp'].dt.hour
df_prep['day_of_week'] = df_prep['timestamp'].dt.dayofweek
df_prep['is_weekend'] = df_prep['day_of_week'].isin([5, 6]).astype(int)

numerical_features = [
    'speed', 'course', 'frequency', 'bandwidth', 'power', 
    'signal_strength', 'location_lat', 'location_lon',
    'hour', 'day_of_week'
]

categorical_features = [
    'status', 'ais_active', 'modulation', 'type', 'flag'
]

targets = ['geohash', 'name', 'type', 'flag', 'destination']

# ====================== 4. PRÉTRAITEMENT ======================

# Encodage des cibles (pour la prédiction)
target_encoders = {}
y = pd.DataFrame()

for target in targets:
    target_encoders[target] = LabelEncoder()
    y[target] = target_encoders[target].fit_transform(df_prep[target])

# Features
X = df_prep[numerical_features + categorical_features + ['geohash']]


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('geohash', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['geohash'])
    ])

X_preprocessed = preprocessor.fit_transform(X)

print(f"Shape après préprocessing : {X_preprocessed.shape}")
print(f"Nombre de features après encodage : {X_preprocessed.shape[1]}")

Shape après préprocessing : (20, 54)
Nombre de features après encodage : 54
